Model 1 : Logistic Regression

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef
)

# ---------------------------------------------------------
# 1) Download dataset using kagglehub (required)
# ---------------------------------------------------------
dataset_path = kagglehub.dataset_download("bhargavadepu/india-waterborne-disease-dataset")
print("Path to dataset files:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(f"No CSV file found inside: {dataset_path}")

csv_path = max(csv_files, key=os.path.getsize)
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)

# ---------------------------------------------------------
# 2) Target selection
# ---------------------------------------------------------
TARGET_COL = "disease"
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Columns: {list(df.columns)}")

# Drop rows with missing target
df = df.dropna(subset=[TARGET_COL])

# Clean target text (if it is string-like)
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

print("\nTarget distribution (top 20):")
print(df[TARGET_COL].value_counts().head(20))

# ---------------------------------------------------------
# 3) Drop obvious leakage / ID-like columns (if present)
# ---------------------------------------------------------
# If a dataset has any ID columns, remove them.
id_like = [c for c in df.columns if c.lower() in {"id", "patient_id", "record_id"} or c.lower().endswith("_id")]
df = df.drop(columns=id_like, errors="ignore")

# OPTIONAL: If there are columns that are literally symptoms of the disease,
# and you want a more "risk factor" style model, you could drop symptom_* columns.
# For now, keep them to maximize predictive power.
# df = df.drop(columns=[c for c in df.columns if c.startswith("symptom_")], errors="ignore")

# ---------------------------------------------------------
# 4) Split X/y
# ---------------------------------------------------------
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# ---------------------------------------------------------
# 5) Train/test split (stratified)
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# 6) Preprocessing: numeric + categorical
# ---------------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---------------------------------------------------------
# 7) Logistic Regression (multiclass)
# NOTE: we do NOT use multi_class=... because your sklearn build rejected it earlier.
# ---------------------------------------------------------
clf = LogisticRegression(
    solver="lbfgs",
    max_iter=6000,
    class_weight="balanced"
)

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", clf)
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# ---------------------------------------------------------
# 8) Evaluation metrics
# ---------------------------------------------------------
acc = accuracy_score(y_test, y_pred)

# Multiclass weighted metrics
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

# Multiclass AUC (OvR weighted)
auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

# MCC supports multiclass
mcc = matthews_corrcoef(y_test, y_pred)

print("\n=== Logistic Regression Results (India Waterborne Disease) ===")
print(f"Target: {TARGET_COL}")
print(f"Classes: {list(model.named_steps['clf'].classes_)}")
print(f"Accuracy : {acc:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"MCC      : {mcc:.4f}")


/Users/sukratisaxena/Desktop/ML Assignment 2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1
Using CSV: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1/waterborne_disease_dataset.csv

Target distribution (top 20):
disease
No_Disease       2101546
Typhoid           629079
Giardiasis        525509
Dysentery         524471
Cholera           419811
Hepatitis_A       419647
Hepatitis_E       367253
Leptospirosis     262684
Name: count, dtype: int64

=== Logistic Regression Results (India Waterborne Disease) ===
Target: disease
Classes: ['Cholera', 'Dysentery', 'Giardiasis', 'Hepatitis_A', 'Hepatitis_E', 'Leptospirosis', 'No_Disease', 'Typhoid']
Accuracy : 0.8882
AUC      : 0.9909
Precision: 0.8882
Recall   : 0.8882
F1 Score : 0.8878
MCC      : 0.8578


Model 2 : Decesion Tree Classifier

In [18]:
import os
import glob
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef
)

# ---------------------------------------------------------
# 1) Download dataset using kagglehub (required)
# ---------------------------------------------------------
dataset_path = kagglehub.dataset_download("bhargavadepu/india-waterborne-disease-dataset")
print("Path to dataset files:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(f"No CSV file found inside: {dataset_path}")

csv_path = max(csv_files, key=os.path.getsize)
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)

# ---------------------------------------------------------
# 2) Target selection
# ---------------------------------------------------------
TARGET_COL = "disease"
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Columns: {list(df.columns)}")

df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

print("\nTarget distribution (top 20):")
print(df[TARGET_COL].value_counts().head(20))

# ---------------------------------------------------------
# 3) Drop ID-like columns (if present)
# ---------------------------------------------------------
id_like = [c for c in df.columns if c.lower() in {"id", "patient_id", "record_id"} or c.lower().endswith("_id")]
df = df.drop(columns=id_like, errors="ignore")

# If you want "risk-factor only" prediction, uncomment this:
# df = df.drop(columns=[c for c in df.columns if c.startswith("symptom_")], errors="ignore")

# ---------------------------------------------------------
# 4) Split X/y
# ---------------------------------------------------------
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# ---------------------------------------------------------
# 5) Train/test split (stratified)
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# 6) Preprocessing: numeric + categorical
# Decision trees don't need scaling. We just impute + one-hot.
# ---------------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---------------------------------------------------------
# 7) Decision Tree Classifier
# Add mild regularization to reduce overfitting
# ---------------------------------------------------------
clf = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=18,          # you can tweak this (e.g., 12, 16, 20)
    min_samples_split=50,
    min_samples_leaf=25
)

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", clf)
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# ---------------------------------------------------------
# 8) Metrics
# ---------------------------------------------------------
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
mcc = matthews_corrcoef(y_test, y_pred)

# Multiclass AUC (OvR weighted)
auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

print("\n=== Decision Tree Results (India Waterborne Disease) ===")
print(f"Target: {TARGET_COL}")
print(f"Classes: {list(model.named_steps['clf'].classes_)}")
print(f"Accuracy : {acc:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"MCC      : {mcc:.4f}")


Path to dataset files: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1
Using CSV: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1/waterborne_disease_dataset.csv

Target distribution (top 20):
disease
No_Disease       2101546
Typhoid           629079
Giardiasis        525509
Dysentery         524471
Cholera           419811
Hepatitis_A       419647
Hepatitis_E       367253
Leptospirosis     262684
Name: count, dtype: int64

=== Decision Tree Results (India Waterborne Disease) ===
Target: disease
Classes: ['Cholera', 'Dysentery', 'Giardiasis', 'Hepatitis_A', 'Hepatitis_E', 'Leptospirosis', 'No_Disease', 'Typhoid']
Accuracy : 0.9238
AUC      : 0.9931
Precision: 0.9256
Recall   : 0.9238
F1 Score : 0.9243
MCC      : 0.9031


Model 3 : K-Nearest Neighbpurs Model 

In [19]:
import os
import glob
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef
)

# ---------------------------------------------------------
# 1) Download dataset using kagglehub (required)
# ---------------------------------------------------------
dataset_path = kagglehub.dataset_download("bhargavadepu/india-waterborne-disease-dataset")
print("Path to dataset files:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(f"No CSV file found inside: {dataset_path}")

csv_path = max(csv_files, key=os.path.getsize)
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)

# ---------------------------------------------------------
# 2) Target selection
# ---------------------------------------------------------
TARGET_COL = "disease"
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Columns: {list(df.columns)}")

df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

print("\nTarget distribution (top 20):")
print(df[TARGET_COL].value_counts().head(20))

# ---------------------------------------------------------
# 3) Drop ID-like columns (if present)
# ---------------------------------------------------------
id_like = [c for c in df.columns if c.lower() in {"id", "patient_id", "record_id"} or c.lower().endswith("_id")]
df = df.drop(columns=id_like, errors="ignore")

# Optional: if you want risk-factor-only model, uncomment:
# df = df.drop(columns=[c for c in df.columns if c.startswith("symptom_")], errors="ignore")

# ---------------------------------------------------------
# 4) (Optional) Subsample for speed
# KNN can be slow on millions of rows. Subsample if needed.
# ---------------------------------------------------------
USE_SUBSAMPLE = True
SUBSAMPLE_N = 300_000   # adjust: 100k–500k depending on your machine

if USE_SUBSAMPLE and len(df) > SUBSAMPLE_N:
    df = df.sample(n=SUBSAMPLE_N, random_state=42)
    print(f"\nUsing subsample for KNN: {SUBSAMPLE_N} rows")

# ---------------------------------------------------------
# 5) Split X/y
# ---------------------------------------------------------
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# 6) Preprocessing: numeric + categorical (KNN needs scaling)
# ---------------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---------------------------------------------------------
# 7) KNN Classifier
# ---------------------------------------------------------
clf = KNeighborsClassifier(
    n_neighbors=15,
    weights="distance",
    n_jobs=-1
)

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", clf)
])

model.fit(X_train, y_train)

# ---------------------------------------------------------
# 8) Metrics
# ---------------------------------------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
mcc = matthews_corrcoef(y_test, y_pred)

auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

print("\n=== KNN Results (India Waterborne Disease) ===")
print(f"Target: {TARGET_COL}")
print(f"Classes: {list(model.named_steps['clf'].classes_)}")
print(f"Accuracy : {acc:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"MCC      : {mcc:.4f}")


Path to dataset files: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1
Using CSV: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1/waterborne_disease_dataset.csv

Target distribution (top 20):
disease
No_Disease       2101546
Typhoid           629079
Giardiasis        525509
Dysentery         524471
Cholera           419811
Hepatitis_A       419647
Hepatitis_E       367253
Leptospirosis     262684
Name: count, dtype: int64

Using subsample for KNN: 300000 rows

=== KNN Results (India Waterborne Disease) ===
Target: disease
Classes: ['Cholera', 'Dysentery', 'Giardiasis', 'Hepatitis_A', 'Hepatitis_E', 'Leptospirosis', 'No_Disease', 'Typhoid']
Accuracy : 0.8604
AUC      : 0.9769
Precision: 0.8601
Recall   : 0.8604
F1 Score : 0.8589
MCC      : 0.8224


In [1]:
import os
import glob
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef
)

# ---------------------------------------------------------
# 0) OneHotEncoder compatible across sklearn versions
# ---------------------------------------------------------
def make_onehot_sparse():
    # We want SPARSE output to avoid RAM blow-ups
    try:
        # sklearn >= 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        # older sklearn
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

# ---------------------------------------------------------
# 1) Download dataset using kagglehub (required)
# ---------------------------------------------------------
dataset_path = kagglehub.dataset_download("bhargavadepu/india-waterborne-disease-dataset")
print("Path to dataset files:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(f"No CSV file found inside: {dataset_path}")

csv_path = max(csv_files, key=os.path.getsize)
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)

# ---------------------------------------------------------
# 2) Target selection
# ---------------------------------------------------------
TARGET_COL = "disease"
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Columns: {list(df.columns)}")

df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

print("\nTarget distribution (top 20):")
print(df[TARGET_COL].value_counts().head(20))

# ---------------------------------------------------------
# 3) Drop ID-like columns (if present)
# ---------------------------------------------------------
id_like = [c for c in df.columns if c.lower() in {"id", "patient_id", "record_id"} or c.lower().endswith("_id")]
df = df.drop(columns=id_like, errors="ignore")

# Optional: risk-factor-only (uncomment if needed)
# df = df.drop(columns=[c for c in df.columns if c.startswith("symptom_")], errors="ignore")

# ---------------------------------------------------------
# 4) Subsample (recommended for NB speed; not strictly required)
# NB is fast, but the dataset is huge; 500k is plenty for stable metrics.
# ---------------------------------------------------------
USE_SUBSAMPLE = True
SUBSAMPLE_N = 500_000

if USE_SUBSAMPLE and len(df) > SUBSAMPLE_N:
    df = df.sample(n=SUBSAMPLE_N, random_state=42)
    print(f"\nUsing subsample for Naive Bayes: {SUBSAMPLE_N} rows")

# ---------------------------------------------------------
# 5) Split X/y
# ---------------------------------------------------------
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# 6) Preprocessing (sparse-safe)
# - Numeric: impute + MaxAbsScaler (keeps sparse compatibility)
# - Categorical: impute + OneHotEncoder (sparse)
# ---------------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MaxAbsScaler())  # works with sparse downstream
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_onehot_sparse())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.3  # encourages sparse output
)

# MultinomialNB expects non-negative features.
# This transformer clips any negative numeric values to 0 after preprocessing.
clip_nonneg = FunctionTransformer(lambda X: X.maximum(0), accept_sparse=True)

# ---------------------------------------------------------
# 7) Multinomial Naive Bayes
# ---------------------------------------------------------
clf = MultinomialNB(alpha=1.0)

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clip_nonneg", clip_nonneg),
    ("clf", clf)
])

model.fit(X_train, y_train)

# ---------------------------------------------------------
# 8) Metrics
# ---------------------------------------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
mcc = matthews_corrcoef(y_test, y_pred)

auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

print("\n=== Naive Bayes Results (MultinomialNB, sparse-safe) ===")
print(f"Target: {TARGET_COL}")
print(f"Classes: {list(model.named_steps['clf'].classes_)}")
print(f"Accuracy : {acc:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"MCC      : {mcc:.4f}")


/Users/sukratisaxena/Desktop/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1
Using CSV: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1/waterborne_disease_dataset.csv

Target distribution (top 20):
disease
No_Disease       2101546
Typhoid           629079
Giardiasis        525509
Dysentery         524471
Cholera           419811
Hepatitis_A       419647
Hepatitis_E       367253
Leptospirosis     262684
Name: count, dtype: int64

Using subsample for Naive Bayes: 500000 rows

=== Naive Bayes Results (MultinomialNB, sparse-safe) ===
Target: disease
Classes: [np.str_('Cholera'), np.str_('Dysentery'), np.str_('Giardiasis'), np.str_('Hepatitis_A'), np.str_('Hepatitis_E'), np.str_('Leptospirosis'), np.str_('No_Disease'), np.str_('Typhoid')]
Accuracy : 0.8181
AUC      : 0.9762
Precision: 0.8205
Recall   : 0.8181
F1 Score : 0.8170
MCC      : 0.7687


In [2]:
import os
import glob
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef
)

# ---------------------------------------------------------
# 0) OneHotEncoder compatible across sklearn versions (sparse)
# ---------------------------------------------------------
def make_onehot_sparse():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

# ---------------------------------------------------------
# 1) Download dataset using kagglehub (required)
# ---------------------------------------------------------
dataset_path = kagglehub.dataset_download("bhargavadepu/india-waterborne-disease-dataset")
print("Path to dataset files:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(f"No CSV file found inside: {dataset_path}")

csv_path = max(csv_files, key=os.path.getsize)
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)

# ---------------------------------------------------------
# 2) Target selection
# ---------------------------------------------------------
TARGET_COL = "disease"
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Columns: {list(df.columns)}")

df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

print("\nTarget distribution (top 20):")
print(df[TARGET_COL].value_counts().head(20))

# ---------------------------------------------------------
# 3) Drop ID-like columns (if present)
# ---------------------------------------------------------
id_like = [c for c in df.columns if c.lower() in {"id", "patient_id", "record_id"} or c.lower().endswith("_id")]
df = df.drop(columns=id_like, errors="ignore")

# Optional: risk-factor-only model
# df = df.drop(columns=[c for c in df.columns if c.startswith("symptom_")], errors="ignore")

# ---------------------------------------------------------
# 4) Subsample for speed (recommended for Random Forest)
# ---------------------------------------------------------
USE_SUBSAMPLE = True
SUBSAMPLE_N = 500_000

if USE_SUBSAMPLE and len(df) > SUBSAMPLE_N:
    df = df.sample(n=SUBSAMPLE_N, random_state=42)
    print(f"\nUsing subsample for Random Forest: {SUBSAMPLE_N} rows")

# ---------------------------------------------------------
# 5) Split X/y
# ---------------------------------------------------------
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# 6) Preprocessing: numeric + categorical
# Random Forest does NOT need scaling. Keep sparse one-hot for categoricals.
# ---------------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_onehot_sparse())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

# ---------------------------------------------------------
# 7) Random Forest (ensemble)
# ---------------------------------------------------------
clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    max_depth=22,
    min_samples_split=50,
    min_samples_leaf=25
)

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", clf)
])

model.fit(X_train, y_train)

# ---------------------------------------------------------
# 8) Metrics
# ---------------------------------------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
mcc = matthews_corrcoef(y_test, y_pred)

auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

print("\n=== Random Forest Results (India Waterborne Disease) ===")
print(f"Target: {TARGET_COL}")
print(f"Classes: {list(model.named_steps['clf'].classes_)}")
print(f"Accuracy : {acc:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"MCC      : {mcc:.4f}")


Path to dataset files: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1
Using CSV: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1/waterborne_disease_dataset.csv

Target distribution (top 20):
disease
No_Disease       2101546
Typhoid           629079
Giardiasis        525509
Dysentery         524471
Cholera           419811
Hepatitis_A       419647
Hepatitis_E       367253
Leptospirosis     262684
Name: count, dtype: int64

Using subsample for Random Forest: 500000 rows

=== Random Forest Results (India Waterborne Disease) ===
Target: disease
Classes: ['Cholera', 'Dysentery', 'Giardiasis', 'Hepatitis_A', 'Hepatitis_E', 'Leptospirosis', 'No_Disease', 'Typhoid']
Accuracy : 0.9159
AUC      : 0.9938
Precision: 0.9171
Recall   : 0.9159
F1 Score : 0.9152
MCC      : 0.8934


In [5]:
import os
import glob
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef
)

from xgboost import XGBClassifier

# ---------------------------------------------------------
# 0) OneHotEncoder compatible across sklearn versions (sparse)
# ---------------------------------------------------------
def make_onehot_sparse():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

# ---------------------------------------------------------
# 1) Download dataset using kagglehub (required)
# ---------------------------------------------------------
dataset_path = kagglehub.dataset_download("bhargavadepu/india-waterborne-disease-dataset")
print("Path to dataset files:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(f"No CSV file found inside: {dataset_path}")

csv_path = max(csv_files, key=os.path.getsize)
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)

# ---------------------------------------------------------
# 2) Target selection
# ---------------------------------------------------------
TARGET_COL = "disease"
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Columns: {list(df.columns)}")

df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

print("\nTarget distribution (top 20):")
print(df[TARGET_COL].value_counts().head(20))

# ---------------------------------------------------------
# 3) Drop ID-like columns (if present)
# ---------------------------------------------------------
id_like = [c for c in df.columns if c.lower() in {"id", "patient_id", "record_id"} or c.lower().endswith("_id")]
df = df.drop(columns=id_like, errors="ignore")

# Optional: risk-factor-only model (uncomment if needed)
# df = df.drop(columns=[c for c in df.columns if c.startswith("symptom_")], errors="ignore")

# ---------------------------------------------------------
# 4) Subsample for speed (recommended)
# ---------------------------------------------------------
USE_SUBSAMPLE = True
SUBSAMPLE_N = 500_000

if USE_SUBSAMPLE and len(df) > SUBSAMPLE_N:
    df = df.sample(n=SUBSAMPLE_N, random_state=42)
    print(f"\nUsing subsample for XGBoost: {SUBSAMPLE_N} rows")

# ---------------------------------------------------------
# 5) Split X/y
# ---------------------------------------------------------
X = df.drop(columns=[TARGET_COL])
y_str = df[TARGET_COL].values

# Encode y to 0..K-1 for XGBoost
le = LabelEncoder()
y = le.fit_transform(y_str)
class_names = list(le.classes_)
print("\nEncoded classes:", {i: name for i, name in enumerate(class_names)})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# 6) Preprocessing: numeric + categorical
# ---------------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_onehot_sparse())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

# ---------------------------------------------------------
# 7) XGBoost (multiclass)
# IMPORTANT: set num_class for safety
# ---------------------------------------------------------
num_class = len(class_names)

clf = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=num_class,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", clf)
])

model.fit(X_train, y_train)

# ---------------------------------------------------------
# 8) Metrics
# ---------------------------------------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
mcc = matthews_corrcoef(y_test, y_pred)

auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

print("\n=== XGBoost Results (India Waterborne Disease) ===")
print("Target: disease")
print("Classes:", class_names)
print(f"Accuracy : {acc:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"MCC      : {mcc:.4f}")


Path to dataset files: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1
Using CSV: /Users/sukratisaxena/.cache/kagglehub/datasets/bhargavadepu/india-waterborne-disease-dataset/versions/1/waterborne_disease_dataset.csv

Target distribution (top 20):
disease
No_Disease       2101546
Typhoid           629079
Giardiasis        525509
Dysentery         524471
Cholera           419811
Hepatitis_A       419647
Hepatitis_E       367253
Leptospirosis     262684
Name: count, dtype: int64

Using subsample for XGBoost: 500000 rows

Encoded classes: {0: 'Cholera', 1: 'Dysentery', 2: 'Giardiasis', 3: 'Hepatitis_A', 4: 'Hepatitis_E', 5: 'Leptospirosis', 6: 'No_Disease', 7: 'Typhoid'}

=== XGBoost Results (India Waterborne Disease) ===
Target: disease
Classes: ['Cholera', 'Dysentery', 'Giardiasis', 'Hepatitis_A', 'Hepatitis_E', 'Leptospirosis', 'No_Disease', 'Typhoid']
Accuracy : 0.9517
AUC      : 0.9981
Precision: 0.9515
Recall   : 0.9517
F1 Scor